In [6]:
import numpy as np
import pandas as pd

data_frame = pd.read_csv('data/housing.csv')
df = data_frame.copy()
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [7]:
df.shape, df.columns.tolist()

((20640, 10),
 ['longitude',
  'latitude',
  'housing_median_age',
  'total_rooms',
  'total_bedrooms',
  'population',
  'households',
  'median_income',
  'median_house_value',
  'ocean_proximity'])

# 1.13 Algorithms (Linear Regression + SVR) — California Housing

Goal:
- Identify the target column (y)
- Train **Linear Regression** as a baseline
- Train **SVR** with different hyperparameters (GridSearchCV)
- Compare using **MAE**, **RMSE**, and **R²**

## Step 1 — Identify the target column
For the common California housing dataset, the target is usually `median_house_value`.

In [8]:
# Set target column (edit here if your file uses another name)
target_candidates = ['median_house_value', 'target', 'Target', 'label', 'y']
target = next((c for c in target_candidates if c in df.columns), None)

if target is None:
    raise ValueError('Target column not found. Please set target manually.')

target

'median_house_value'

## Step 2 — Train/Test split

In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=[target]).copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

((16512, 9), (4128, 9))

## Step 3 — Preprocessing (simple + robust)
We handle missing values, one-hot encode categorical columns, and scale numeric columns.

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features),
    ],
    remainder='drop'
)

(len(num_features), len(cat_features), cat_features)

(8, 1, ['ocean_proximity'])

## Step 4 — Baseline: Linear Regression

In [12]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

lr_model = Pipeline([
    ('preprocess', preprocess),
    ('model', LinearRegression()),
])

lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

lr_metrics = {
    'model': 'LinearRegression',
    'MAE': mean_absolute_error(y_test, y_pred_lr),
    'RMSE': root_mean_squared_error(y_test, y_pred_lr),
    'R2': r2_score(y_test, y_pred_lr),
}

lr_metrics

{'model': 'LinearRegression',
 'MAE': 50670.48923565362,
 'RMSE': 70059.19333925015,
 'R2': 0.6254382675296266}

## Step 5 — SVR (with hyperparameter tuning)
We tune `kernel`, `C`, `epsilon`, and `gamma` using GridSearchCV.

In [13]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR

svr_pipe = Pipeline([
    ('preprocess', preprocess),
    ('model', SVR()),
])

param_grid = {
    'model__kernel': ['rbf', 'linear'],
    'model__C': [1, 10, 100],
    'model__epsilon': [0.1, 0.2, 0.5],
    'model__gamma': ['scale', 'auto'],
}

grid = GridSearchCV(
    svr_pipe,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)
grid.fit(X_train, y_train)

grid.best_params_

{'model__C': 100,
 'model__epsilon': 0.5,
 'model__gamma': 'scale',
 'model__kernel': 'linear'}

In [14]:
best_svr = grid.best_estimator_
y_pred_svr = best_svr.predict(X_test)

svr_metrics = {
    'model': 'SVR (tuned)',
    'MAE': mean_absolute_error(y_test, y_pred_svr),
    'RMSE': root_mean_squared_error(y_test, y_pred_svr),
    'R2': r2_score(y_test, y_pred_svr),
}

svr_metrics

{'model': 'SVR (tuned)',
 'MAE': 49552.50830520122,
 'RMSE': 72169.53770147431,
 'R2': 0.6025330825315038}

## Step 6 — Compare models

In [15]:
results = pd.DataFrame([lr_metrics, svr_metrics]).sort_values('RMSE')
results

,model,MAE,RMSE,R2
0,LinearRegression,50670.489236,70059.193339,0.625438
1,SVR (tuned),49552.508305,72169.537701,0.602533


## (Optional) Save the best model
We save the best model (lowest RMSE) as a `joblib` file.

In [16]:
import joblib

best_model = best_svr if svr_metrics['RMSE'] <= lr_metrics['RMSE'] else lr_model
model_path = 'output/1.13_best_regression_model.joblib'
joblib.dump(best_model, model_path)
model_path

'output/1.13_best_regression_model.joblib'